# v8 Brain Tumor Segmentation Training on Google Colab

**What this notebook does:** Trains the v8 brain tumor segmentation model (ConvNeXt-Tiny + 384px + Tversky loss + 50/50 balanced sampler + crash-safe checkpointing) on a free Colab T4 / paid A100 / L4 GPU.

**Why Colab:** Your laptop hit 112.8°C peak CPU temp during local training. Colab gives you a properly-cooled cloud GPU without thermal risk. Free tier (T4) does the job; Pro tier (A100/L4) finishes 2-3x faster.

**Estimated training time:**
- **T4 (free)**: ~24 hours total. Free Colab sessions disconnect after ~12 hours — the trainer's `--resume auto` handles this. You'll reconnect 2-3 times.
- **L4 (Pro)**: ~12 hours. Single Pro session.
- **A100 (Pro+)**: ~6 hours. Single Pro+ session.

**Before you start:**
1. Upload `dataset_v8.zip` to your Google Drive at `MyDrive/neurolens/dataset_v8.zip` (~860 MB)
2. Upload `colab_bundle.zip` to your Google Drive at `MyDrive/neurolens/colab_bundle.zip` (~30 KB)
3. Set runtime: `Runtime → Change runtime type → GPU → T4/L4/A100`
4. Run all cells below in order

## 1. Verify GPU

In [ ]:
!nvidia-smi

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Extract dataset_v8 from Drive to local Colab disk

Local disk has much faster I/O than mounted Drive. Once extracted (~5 min), training reads from local.

In [ ]:
import os, time
DRIVE_ROOT = '/content/drive/MyDrive/neurolens'
LOCAL_ROOT = '/content/neurolens'
os.makedirs(LOCAL_ROOT, exist_ok=True)

if not os.path.isdir(f'{LOCAL_ROOT}/dataset_v8/train/images'):
    print('Extracting dataset_v8.zip from Drive to local /content...')
    t0 = time.time()
    !unzip -q '{DRIVE_ROOT}/dataset_v8.zip' -d {LOCAL_ROOT}/
    print(f'Extracted in {time.time() - t0:.0f}s')
else:
    print('dataset_v8 already present, skipping extraction')

!ls {LOCAL_ROOT}/dataset_v8/train/images | wc -l
!ls {LOCAL_ROOT}/dataset_v8/val/images | wc -l
!ls {LOCAL_ROOT}/dataset_v8/test/images | wc -l

## 4. Extract code bundle

In [ ]:
if not os.path.isdir(f'{LOCAL_ROOT}/src'):
    !unzip -q '{DRIVE_ROOT}/colab_bundle.zip' -d {LOCAL_ROOT}/
    !ls {LOCAL_ROOT}/src/
else:
    print('Code bundle already present')

## 5. Install dependencies

Colab pre-installs torch, torchvision, numpy, pillow, scikit-image. We only add segmentation-models-pytorch and timm.

In [ ]:
!pip install -q segmentation-models-pytorch>=0.3.3 timm>=0.9.16
import torch, torchvision
print(f'torch={torch.__version__}, torchvision={torchvision.__version__}, cuda={torch.cuda.is_available()}, device={torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')

## 6. Set checkpoint path on Drive (survives Colab disconnects)

The trainer writes checkpoints to a Drive directory. When Colab disconnects, you reconnect and the trainer's `--resume auto` picks up exactly where it left off.

In [ ]:
CHECKPOINT_DIR = f'{DRIVE_ROOT}/attention_unet_v8'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f'Checkpoint dir (on Drive): {CHECKPOINT_DIR}')
!ls -la {CHECKPOINT_DIR}/ 2>&1 || echo '(empty, fresh training)'

## 7. Launch training

**Run this cell to train.** On disconnect, just rerun this cell — `--resume auto` continues from `last.pt` in CHECKPOINT_DIR.

**Tweakable knobs:**
- `--batch_size 4` (T4 may need 2; A100 can do 8-16)
- `--epochs 60` (drop to 30 for a half-training run)
- `--image_size 384` (drop to 256 if VRAM tight)
- `--checkpoint_every_steps 500` (lower for safer crash recovery, higher for less Drive I/O)

Output streams live so you can watch each epoch.

In [ ]:
import sys
sys.path.insert(0, f'{LOCAL_ROOT}')

# T4 hint: if you get CUDA OOM at batch=4, reduce to --batch_size 2.
# A100 hint: --batch_size 8 fits comfortably and trains 2x faster.

!cd {LOCAL_ROOT} && python src/train_segmentation_v7.py \
    --data_dir {LOCAL_ROOT}/dataset_v8 \
    --output_dir '{CHECKPOINT_DIR}' \
    --epochs 60 \
    --batch_size 4 \
    --image_size 384 \
    --num_workers 2 \
    --checkpoint_every_steps 500 \
    --resume auto

## 8. Monitor GPU + temperatures during training (optional)

Open this cell in a separate cell while training runs to watch GPU usage.

In [ ]:
!nvidia-smi --query-gpu=name,temperature.gpu,power.draw,memory.used,memory.total,utilization.gpu --format=csv

## 9. After training: export to ONNX (optional)

Once `best_micro.pt` exists in the checkpoint dir, export to ONNX for fast inference and the HF Space.

In [ ]:
import torch, segmentation_models_pytorch as smp
ckpt_path = f'{CHECKPOINT_DIR}/best_micro.pt'
ckpt = torch.load(ckpt_path, map_location='cuda', weights_only=False)
model = smp.Unet(encoder_name='tu-convnext_tiny.fb_in22k_ft_in1k', encoder_weights=None, in_channels=3, classes=1).cuda().eval()
model.load_state_dict(ckpt['model_state_dict'])
dummy = torch.randn(1, 3, 384, 384, device='cuda')
onnx_path = f'{CHECKPOINT_DIR}/best_micro.onnx'
torch.onnx.export(model, dummy, onnx_path, input_names=['input'], output_names=['output'],
    dynamic_axes={'input': {0: 'batch'}, 'output': {0: 'batch'}}, opset_version=17)
print(f'Exported to {onnx_path}')
!ls -lh {onnx_path}

## 10. Download final checkpoints

The checkpoint dir is already on your Drive. You can:
1. Download from Drive web UI: drive.google.com → neurolens/attention_unet_v8
2. Or use the cell below to ZIP and prepare for download

In [ ]:
!cd '{CHECKPOINT_DIR}' && zip -r /content/v8_final.zip best_micro.pt best_model.pt last.pt training.log 2>&1 | tail -10
from google.colab import files
files.download('/content/v8_final.zip')

---

## Troubleshooting

### Colab disconnected mid-training
Rerun cell 7. `--resume auto` picks up from `last.pt` in Drive. No work lost beyond the last `--checkpoint_every_steps 500` save (max ~2-5 min on T4).

### CUDA out of memory
Lower `--batch_size` to 2, then 1. Or lower `--image_size` to 320 or 256. T4 has 15 GB VRAM; 384 × batch 4 fits but tight.

### Slow Drive I/O
If extraction is slow, the dataset is on Drive in zip form (single file = fast). If you bypassed step 3 and read directly from Drive (`--data_dir /content/drive/MyDrive/neurolens/dataset_v8`), Drive I/O bottlenecks training. Always extract to /content first.

### Want to inspect a checkpoint without retraining
```python
ckpt = torch.load(f'{CHECKPOINT_DIR}/best_micro.pt', weights_only=False, map_location='cpu')
print('Epoch:', ckpt['epoch'])
print('Best micro_dice:', ckpt['best_micro'])
print('Best composite:', ckpt['best_composite'])
print('Args:', ckpt['args'])
```

### Want to compare two checkpoints
After training finishes, you'll have `best_micro.pt` (highest micro-Dice, the landing-page number you want) and `best_model.pt` (highest composite = dice - 5·fp_rate, production-safest FP discipline). Pick `best_micro.pt` for headline numbers, `best_model.pt` for deployment.